# ARENA3 PPO — P2 (S2A_pre 출발, PPO 미세조정)

교수님이 제안한 Stable-Baselines3 계열의 PPO를 지금까지와 같은 조건에서 시험합니다.
- 같은 것: 관측 119칸, 6모드 행동(갈림길에서만 고름), 정확히 정산한 지연 보상, G1과 같은 학습 상황, 같은 심판 84상황.
- 다른 것: 학습 규칙만 PPO(SB3 기본값: 클립 0.2, GAE 0.95, 에포크 10, 미니배치 64, 학습률 3e-4).
- 같은 난수의 **S2A_pre(사전학습만 한 모델)에서 출발**합니다. 처음 탐욕 선택은 S2A_pre와 완전히 같습니다(맥에서 1,802지점 확인).
묻는 것: **PPO 미세조정이 사전학습 위에 이득을 더하는가**(2단계 DQN 미세조정은 더하지 못했습니다).
- 예산: 난수마다 에피소드 448판(G1 수준). 시간이 아니라 **판 수**로 셉니다.

1. 런타임 유형을 **CPU + 높은 RAM**(CPU 8개)으로 고르고 **런타임 → 모두 실행**을 누르세요. 첫 칸이 CPU 수를 확인합니다. 다른 PPO 노트북과 **동시에** 돌리면 됩니다.
2. 순서: 실행 전 점검(진짜 SUMO 4판, 약 10분) → 학습 난수 3개 동시(약 10시간) → 심판 평가 252판(약 1.5시간) → 참고 판정.
3. 끊기면 처음 네 칸을 실행한 뒤 멈춘 칸부터 다시 실행하세요(또는 **모두 실행**). 끝난 단계는 건너뛰고, 학습은 체크포인트에서 이어집니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_PPO.zip"
A3 = DRIVE / "arena3"
VARIANT = "P2"
OUT = A3 / f"ppo_{VARIANT}"           # 런타임마다 다른 폴더(두 런타임이 같은 이름 폴더를 동시에 만들지 않게)
S2A_TRAIN = A3 / "stage2" / "S2A_train"  # P2의 출발 모델(S2A_pre)
WORKSPACE = Path("/content/FutureTraffic_ARENA3_PPO")
MIN_CPUS = 8
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 방식:", VARIANT, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요. "
                          "(경험 양은 판 수로 정해져 같지만, CPU 2개면 학습이 약 4배 오래 걸려 24시간 한도에 걸립니다.)")
if VARIANT == "P2":
    missing = [s for s in (42, 43, 44) if not (S2A_TRAIN / f"seed_{s}" / "pretrained" / "model.pt").exists()]
    assert not missing, f"S2A_pre 모델이 없습니다(난수 {missing}): {S2A_TRAIN}"

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_PPO_V1", "ARENA3 PPO 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PPO 설정 {package_manifest['ppo_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검: 진짜 SUMO로 에피소드 4판·갱신 2번(평가자 예열 1번 + PPO 1번)을 점검용 폴더에서 돌립니다(약 10분).
#    본 학습 폴더와 섞이지 않습니다.
SMOKE = OUT / "smoke"
if (SMOKE / "smoke.json").exists():
    print("이전 점검이 통과해 건너뜁니다:", colab.read_json(SMOKE / "smoke.json")["episodes"], "에피소드")
else:
    args = ["src/arena3_ppo.py", "smoke", "--variant", VARIANT, "--seed", "42", "--out", str(SMOKE),
            "--work", "/content/ppo_smoke"]
    if VARIANT == "P2":
        args += ["--init-root", str(S2A_TRAIN)]
    colab.run_live(args, WORKSPACE)

In [ ]:
# 2) 학습: 난수 42·43·44를 동시에(난수마다 에피소드 2판씩 병렬). 난수마다 448판에서 멈춥니다(약 10시간).
#    셀을 멈춰도 학습은 계속 돕니다. 런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하면 체크포인트에서 이어집니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
TRAIN = OUT / "train"
LOGS = OUT / "logs"


def command(seed):
    args = ["src/arena3_ppo.py", "train", "--variant", VARIANT, "--seed", str(seed), "--out", str(TRAIN),
            "--work", f"/content/ppo_{seed}"]
    return args + (["--init-root", str(S2A_TRAIN)] if VARIANT == "P2" else [])


def describe(job):
    s = colab.read_json(TRAIN / f"seed_{job[-2:]}" / "status.json")
    if not s:
        return None
    last = s.get("last_update") or {}
    return (f"에피소드 {s['episodes']}/{s['budget_episodes']}, 갱신 {s['updates']}, 선택 {s['choices']}, "
            f"버린 판 {s['discarded']}, KL {last.get('approx_kl')}, 끝남 {s['finished']}")


pids = colab.launch({f"{VARIANT}_seed{s}": command(s) for s in (42, 43, 44)}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, f"{VARIANT} 학습에서 오류가 났습니다. 위 로그를 저에게 보내 주세요."

In [ ]:
# 3) 심판 평가: 최종 모델 3개 × 84상황 = 252판. 1단계와 같은 심판·같은 84상황입니다. 끊겨도 다시 실행하면 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
TRAIN = OUT / "train"  # 학습 칸(2)을 건너뛰고 이 칸만 실행해도 되도록 여기서도 정한다
missing = [s for s in (42, 43, 44) if not (TRAIN / f"seed_{s}" / "final" / "model.pt").exists()]
assert not missing, f"최종 모델이 아직 없습니다(난수 {missing}). 2)를 다시 실행하세요."
colab.run_live(["src/arena3_ppo.py", "evaluate", "--variant", VARIANT, "--train", str(TRAIN), "--out", str(OUT / "eval"),
                "--work", "/content/ppo_eval", "--workers", str(CPUS)], WORKSPACE)
print(f"{VARIANT} 평가 끝. 다른 PPO 노트북도 끝나면 ARENA3_PPO_judge를 실행하세요.")

In [ ]:
# 4) 참고 판정: 이 방식과 기존 후보(G1·S2A_pre·S2A·S2B_pre·S2B)를 같은 심판으로 봅니다(SUMO 없음, 몇 분).
#    최종 판정과 C1·C2 확인은 ARENA3_PPO_judge 노트북에서 합니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
from IPython.display import display, Markdown
S2 = A3 / "stage2"
args = ["src/arena3_ppo.py", "judge", "--baseline", str(A3 / "referee" / "baseline"),
        "--ppo", f"{VARIANT}={OUT / 'eval'}", "--reference", f"G1={A3 / 'referee' / 'G1'}", "--out", str(OUT / "judge")]
for n in ("S2A_pre", "S2A", "S2B_pre", "S2B"):
    args += ["--reference", f"{n}={S2 / 'eval' / n}"]
colab.run_live(args, WORKSPACE, check=False)
if (OUT / "judge" / "summary.md").exists():
    display(Markdown((OUT / "judge" / "summary.md").read_text()))